In [1]:

import pandas as pd
import numpy as np
import os

# Load the dataset
df = pd.read_csv('perovskite_complete_2014_2026_v2.csv')
print(f"Shape: {df.shape}")
print(f"Columns (first 10): {list(df.columns[:10])}")
print(f"Columns (last 10): {list(df.columns[-10:])}")
print(f"\npub_year distribution:")
print(df['pub_year'].value_counts().sort_index())
print(f"\nPCE stats:")
print(df['PCE'].describe())
print(f"\nPCE missing: {df['PCE'].isna().sum()}")
print(f"pub_year missing: {df['pub_year'].isna().sum()}")


Shape: (7773, 511)
Columns (first 10): ['journal', 'pub_year', 'architecture', 'cell_area', 'composition_short', 'composition_long', 'band_gap', 'perovskite_thickness', 'dimension_3D', 'dimension_layers']
Columns (last 10): ['sub_temp_type', 'sub_temp_main', 'sub_temp_part1', 'sub_thickness_type', 'sub_thickness_main', 'sub_thickness_part1', 'ds_prep_method', 'ds_token_check', 'ds_semantic_annot', 'ds_process_interact']

pub_year distribution:
pub_year
2014      16
2015      39
2016      37
2017      86
2018     134
2019     191
2020     507
2021     982
2022    1211
2023    1491
2024    1973
2025     989
2026     117
Name: count, dtype: int64

PCE stats:
count    7266.000000
mean       17.895997
std         5.726595
min         0.006000
25%        14.862500
50%        19.320000
75%        22.140000
max        27.490000
Name: PCE, dtype: float64

PCE missing: 507
pub_year missing: 0


C:\Users\陈\AppData\Local\Temp\ipykernel_39540\1924507345.py:6: DtypeWarning: Columns (74,76,77,78,79,81,98,100,101,102,103,104,105,150,158,196,201,205,215,230,235,246,249,261,265,269,270,273,276,286,287,288,299,300,301,309,310,322,323,333,343,344,359,363,365,366,376,381,385,394,395,403,404,406,407,413,418,420,432,433,438,439,440,451,491,493,494,496,497,499,500) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('perovskite_complete_2014_2026_v2.csv')


In [2]:

# Drop rows with missing PCE
df = df.dropna(subset=['PCE']).copy()
print(f"After dropping missing PCE: {df.shape}")

# --- 高效时代子集：2022–2025，按年份分层抽样至 4500 条（与 2.ipynb、论文口径一致）---
from sklearn.model_selection import train_test_split
EFF_YEAR_MIN, EFF_YEAR_MAX = 2022, 2025
EFF_TARGET_N = 4500
EFF_RANDOM_STATE = 42
_df_win = df[(df['pub_year'] >= EFF_YEAR_MIN) & (df['pub_year'] <= EFF_YEAR_MAX)].copy()
if len(_df_win) < EFF_TARGET_N:
    raise ValueError(f"窗口 [{EFF_YEAR_MIN},{EFF_YEAR_MAX}] 内有效样本不足 {EFF_TARGET_N}，仅 {len(_df_win)}")
if len(_df_win) > EFF_TARGET_N:
    _df_win, _ = train_test_split(
        _df_win,
        train_size=EFF_TARGET_N,
        random_state=EFF_RANDOM_STATE,
        stratify=_df_win['pub_year'],
    )
df = _df_win.reset_index(drop=True)
print(f"高效时代子集 ({EFF_YEAR_MIN}–{EFF_YEAR_MAX}, stratified n={EFF_TARGET_N}): {df.shape}")
print(df['pub_year'].value_counts().sort_index())

# Identify numeric and non-numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
non_numeric_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()
print(f"Numeric columns: {len(numeric_cols)}")
print(f"Non-numeric columns: {len(non_numeric_cols)}")
print(f"Non-numeric: {non_numeric_cols[:20]}")

# Check if PCE is in numeric cols
print(f"PCE in numeric: {'PCE' in numeric_cols}")
print(f"pub_year in numeric: {'pub_year' in numeric_cols}")

# Feature columns: all numeric except PCE and pub_year (which we'll use as metadata)
feature_cols = [c for c in numeric_cols if c not in ['PCE', 'pub_year']]
print(f"Feature columns count: {len(feature_cols)}")

# Check for any non-numeric features that could be encoded
print(f"\nAll non-numeric cols: {non_numeric_cols}")


After dropping missing PCE: (7266, 511)
高效时代子集 (2022–2025, stratified n=4500): (4500, 511)
pub_year
2022     962
2023    1185
2024    1567
2025     786
Name: count, dtype: int64
Numeric columns: 108
Non-numeric columns: 403
Non-numeric: ['journal', 'architecture', 'composition_short', 'composition_long', 'dimension_3D', 'dimension_layers', 'dimension_2D', 'dimension_2D3D', 'main_solvent', 'sol_vol', 'quenching', 'solvent_annealing', 'etl_stack', 'etl_solvents', 'etl_surf_treat', 'htl_stack', 'htl_solvents', 'htl_main_solvent', 'htl_surf_treat', 'composition_std']
PCE in numeric: True
pub_year in numeric: True
Feature columns count: 106

All non-numeric cols: ['journal', 'architecture', 'composition_short', 'composition_long', 'dimension_3D', 'dimension_layers', 'dimension_2D', 'dimension_2D3D', 'main_solvent', 'sol_vol', 'quenching', 'solvent_annealing', 'etl_stack', 'etl_solvents', 'etl_surf_treat', 'htl_stack', 'htl_solvents', 'htl_main_solvent', 'htl_surf_treat', 'composition_std', 

In [3]:

# Check cardinality of non-numeric columns
cardinality = {}
for col in non_numeric_cols:
    cardinality[col] = df[col].nunique(dropna=True)

card_df = pd.DataFrame(list(cardinality.items()), columns=['column', 'nunique']).sort_values('nunique')
print("Low cardinality categorical columns (<= 20 unique values):")
low_card = card_df[card_df['nunique'] <= 20]
print(low_card.head(30))
print(f"\nTotal low cardinality: {len(low_card)}")

print("\nHigh cardinality columns (> 100 unique values):")
high_card = card_df[card_df['nunique'] > 100]
print(high_card)
print(f"\nTotal high cardinality: {len(high_card)}")

print(f"\nAll cardinality stats: min={card_df['nunique'].min()}, max={card_df['nunique'].max()}, median={card_df['nunique'].median()}")


Low cardinality categorical columns (<= 20 unique values):
                    column  nunique
306    stack_layer_2_shell        0
126              sol_add_5        0
163                 surf_2        0
277        htl_mat_1_shell        0
350            sub_1_part3        1
51             A_formula_5        1
367   sub_clean_step4_main        1
368  sub_clean_step4_part1        1
52              A_source_5        1
50               A_iupac_5        1
92             C_formula_3        1
362  sub_clean_step2_part1        1
361   sub_clean_step2_main        1
356             sub_5_type        1
48                 A_ion_5        1
53             A_src_cas_5        1
90              C_smiles_3        1
91               C_iupac_3        1
89                 C_ion_3        1
93              C_source_3        1
139           htl_conc_s_3        1
287         htl_mat_4_type        1
288         htl_mat_5_type        1
282         htl_mat_2_dop1        1
95             C_src_cas_3        1
305  

In [4]:

# Let's try one-hot encoding for columns with cardinality between 2 and 50
# and frequency encoding for high cardinality

# Select categorical columns to one-hot encode (2-50 unique values)
cols_to_ohe = [col for col in non_numeric_cols if 2 <= cardinality[col] <= 50]
print(f"Columns to one-hot encode (2-50 unique): {len(cols_to_ohe)}")

# Estimate total features after OHE
total_ohe_features = sum(cardinality[col] for col in cols_to_ohe)
print(f"Estimated one-hot features: {total_ohe_features}")

# Frequency encode high cardinality columns (keeping top N maybe)
high_card_cols = [col for col in non_numeric_cols if cardinality[col] > 50]
print(f"High cardinality columns to frequency encode: {len(high_card_cols)}")

# Let's do a simpler but practical approach
# OHE for columns with 2-20 unique values only
cols_to_ohe_simple = [col for col in non_numeric_cols if 2 <= cardinality[col] <= 20]
total_simple = sum(cardinality[col] for col in cols_to_ohe_simple)
print(f"Simple OHE columns (2-20): {len(cols_to_ohe_simple)}, features: {total_simple}")
print(cols_to_ohe_simple[:20])


Columns to one-hot encode (2-50 unique): 251
Estimated one-hot features: 2729
High cardinality columns to frequency encode: 68
Simple OHE columns (2-20): 212, features: 1378
['architecture', 'dimension_3D', 'dimension_layers', 'dimension_2D', 'dimension_2D3D', 'quenching', 'solvent_annealing', 'A_smiles_1', 'A_iupac_1', 'A_formula_1', 'A_source_1', 'A_cas_1', 'A_src_cas_1', 'A_smiles_2', 'A_iupac_2', 'A_formula_2', 'A_source_2', 'A_cas_2', 'A_src_cas_2', 'A_ion_3']


In [5]:

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from lightgbm import LGBMRegressor
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
import warnings
warnings.filterwarnings('ignore')

# Prepare base feature matrix: numeric features only (most reliable)
feature_cols = [c for c in numeric_cols if c not in ['PCE', 'pub_year']]
X_base = df[feature_cols].fillna(0).values
y = df['PCE'].values
years = df['pub_year'].values.astype(int)

print(f"Base feature matrix shape: {X_base.shape}")
print(f"Target shape: {y.shape}")
print(f"Year range: {years.min()} - {years.max()}")

# Also prepare a one-hot encoded version for richer features
# Let's OHE columns with 2-15 unique values (very low cardinality)
very_low_card = [col for col in non_numeric_cols if 2 <= cardinality[col] <= 15]
print(f"Very low cardinality columns for OHE: {len(very_low_card)}")

# Create a small encoded feature set
df_encoded = pd.get_dummies(df[very_low_card], dummy_na=False, drop_first=False)
X_ohe = df_encoded.fillna(0).values
print(f"OHE feature matrix shape: {X_ohe.shape}")

# Combine numeric + OHE
X_combined = np.hstack([X_base, X_ohe])
print(f"Combined feature matrix shape: {X_combined.shape}")

# Use combined as our main feature set
X = X_combined


Base feature matrix shape: (4500, 106)
Target shape: (4500,)
Year range: 2022 - 2025
Very low cardinality columns for OHE: 194
OHE feature matrix shape: (4500, 1045)
Combined feature matrix shape: (4500, 1151)


In [6]:

def uniform_time_split(X, y, years, test_size=0.2, random_state=42):
    """For each year, do an 80/20 random split. Combine all train and all test."""
    train_idx = []
    test_idx = []
    unique_years = sorted(np.unique(years))
    
    for year in unique_years:
        year_mask = years == year
        year_indices = np.where(year_mask)[0]
        if len(year_indices) < 5:
            # For very small years, put all in train
            train_idx.extend(year_indices.tolist())
            continue
        y_year = y[year_indices]
        # Simple random split within year (no stratification since PCE is continuous)
        tr_idx, te_idx = train_test_split(
            year_indices, test_size=test_size, random_state=random_state
        )
        train_idx.extend(tr_idx.tolist())
        test_idx.extend(te_idx.tolist())
    
    return np.array(train_idx), np.array(test_idx)

def evaluate_model(model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    return r2, rmse, mae, y_pred


def uniform_time_kfold_assign(years, n_splits=5, random_state=42):
    """与 uniform_time_split 同年分层：每个 pub_year 内随机打乱后按 round-robin 分到 n_splits 折。
    若某年样本数 < n_splits，则该年样本 fold=-1，仅参与训练、不作验证折（与 uniform_time_split 中小年不进测试集一致）。"""
    rng = np.random.RandomState(random_state)
    fold_id = np.full(len(years), -1, dtype=int)
    for y in np.unique(years):
        idx = np.where(years == y)[0]
        if len(idx) < n_splits:
            fold_id[idx] = -1
            continue
        perm = rng.permutation(idx)
        for slot, sample_idx in enumerate(perm):
            fold_id[sample_idx] = slot % n_splits
    return fold_id


def iter_uniform_time_kfold(years, n_splits=5, random_state=42):
    fold_id = uniform_time_kfold_assign(years, n_splits=n_splits, random_state=random_state)
    for k in range(n_splits):
        te = np.where(fold_id == k)[0]
        tr = np.where(fold_id != k)[0]
        yield k, tr, te


# Test uniform split
for seed in [42, 2024, 31415]:
    tr_idx, te_idx = uniform_time_split(X, y, years, test_size=0.2, random_state=seed)
    print(f"Seed {seed}: train={len(tr_idx)}, test={len(te_idx)}, train_years={sorted(set(years[tr_idx]))}, test_years={sorted(set(years[te_idx]))}")


Seed 42: train=3598, test=902, train_years=[np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)], test_years=[np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Seed 2024: train=3598, test=902, train_years=[np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)], test_years=[np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Seed 31415: train=3598, test=902, train_years=[np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)], test_years=[np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


In [7]:

# Part 1: Full data uniform split baseline
seeds = [42, 2024, 31415]
models = {
    'LightGBM': LGBMRegressor(n_estimators=500, max_depth=8, learning_rate=0.05, random_state=42, verbosity=-1, n_jobs=-1),
    'ExtraTrees': ExtraTreesRegressor(n_estimators=300, max_depth=15, random_state=42, n_jobs=-1),
    'RandomForest': RandomForestRegressor(n_estimators=300, max_depth=15, random_state=42, n_jobs=-1)
}

results_part1 = []

for seed in seeds:
    tr_idx, te_idx = uniform_time_split(X, y, years, test_size=0.2, random_state=seed)
    X_train, X_test = X[tr_idx], X[te_idx]
    y_train, y_test = y[tr_idx], y[te_idx]
    
    # Standardize features
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)
    
    for model_name, model_template in models.items():
        # Create fresh model instance with current seed
        if model_name == 'LightGBM':
            model = LGBMRegressor(n_estimators=500, max_depth=8, learning_rate=0.05, random_state=seed, verbosity=-1, n_jobs=-1)
        elif model_name == 'ExtraTrees':
            model = ExtraTreesRegressor(n_estimators=300, max_depth=15, random_state=seed, n_jobs=-1)
        else:
            model = RandomForestRegressor(n_estimators=300, max_depth=15, random_state=seed, n_jobs=-1)
        
        r2, rmse, mae, y_pred = evaluate_model(model, X_train_s, y_train, X_test_s, y_test)
        results_part1.append({
            'seed': seed,
            'model': model_name,
            'r2': r2,
            'rmse': rmse,
            'mae': mae,
            'n_train': len(tr_idx),
            'n_test': len(te_idx)
        })
        print(f"Seed {seed}, {model_name}: R²={r2:.4f}, RMSE={rmse:.4f}, MAE={mae:.4f}")

print("\n--- Part 1 Complete ---")


Seed 42, LightGBM: R²=0.7598, RMSE=2.8325, MAE=2.1156
Seed 42, ExtraTrees: R²=0.7350, RMSE=2.9752, MAE=2.1493
Seed 42, RandomForest: R²=0.7476, RMSE=2.9032, MAE=2.1553
Seed 2024, LightGBM: R²=0.7479, RMSE=2.8692, MAE=2.0643
Seed 2024, ExtraTrees: R²=0.7139, RMSE=3.0566, MAE=2.2118
Seed 2024, RandomForest: R²=0.7328, RMSE=2.9537, MAE=2.1507
Seed 31415, LightGBM: R²=0.7913, RMSE=2.7257, MAE=2.0179
Seed 31415, ExtraTrees: R²=0.7637, RMSE=2.9002, MAE=2.1304
Seed 31415, RandomForest: R²=0.7805, RMSE=2.7947, MAE=2.0877

--- Part 1 Complete ---


In [8]:

# Use only numeric features for speed (106 dims)
X = X_base  # 106 numeric features only

# Part 1: Full data uniform split baseline with reduced complexity
seeds = [42, 2024, 31415]

results_part1 = []

for seed in seeds:
    tr_idx, te_idx = uniform_time_split(X, y, years, test_size=0.2, random_state=seed)
    X_train, X_test = X[tr_idx], X[te_idx]
    y_train, y_test = y[tr_idx], y[te_idx]
    
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)
    
    # LightGBM
    model = LGBMRegressor(n_estimators=500, max_depth=8, learning_rate=0.05, random_state=seed, verbosity=-1, n_jobs=4)
    r2, rmse, mae, y_pred = evaluate_model(model, X_train_s, y_train, X_test_s, y_test)
    results_part1.append({'seed': seed, 'model': 'LightGBM', 'r2': r2, 'rmse': rmse, 'mae': mae})
    print(f"Seed {seed}, LightGBM: R²={r2:.4f}, RMSE={rmse:.4f}, MAE={mae:.4f}")
    
    # ExtraTrees
    model = ExtraTreesRegressor(n_estimators=300, max_depth=15, random_state=seed, n_jobs=4)
    r2, rmse, mae, y_pred = evaluate_model(model, X_train_s, y_train, X_test_s, y_test)
    results_part1.append({'seed': seed, 'model': 'ExtraTrees', 'r2': r2, 'rmse': rmse, 'mae': mae})
    print(f"Seed {seed}, ExtraTrees: R²={r2:.4f}, RMSE={rmse:.4f}, MAE={mae:.4f}")
    
    # RandomForest
    model = RandomForestRegressor(n_estimators=300, max_depth=15, random_state=seed, n_jobs=4)
    r2, rmse, mae, y_pred = evaluate_model(model, X_train_s, y_train, X_test_s, y_test)
    results_part1.append({'seed': seed, 'model': 'RandomForest', 'r2': r2, 'rmse': rmse, 'mae': mae})
    print(f"Seed {seed}, RandomForest: R²={r2:.4f}, RMSE={rmse:.4f}, MAE={mae:.4f}")
    print("---")


Seed 42, LightGBM: R²=0.6788, RMSE=3.2754, MAE=2.4238
Seed 42, ExtraTrees: R²=0.6386, RMSE=3.4740, MAE=2.5367
Seed 42, RandomForest: R²=0.6715, RMSE=3.3121, MAE=2.4172
---
Seed 2024, LightGBM: R²=0.6768, RMSE=3.2486, MAE=2.3572
Seed 2024, ExtraTrees: R²=0.6546, RMSE=3.3583, MAE=2.4438
Seed 2024, RandomForest: R²=0.6703, RMSE=3.2812, MAE=2.3412
---
Seed 31415, LightGBM: R²=0.7298, RMSE=3.1011, MAE=2.2734
Seed 31415, ExtraTrees: R²=0.7018, RMSE=3.2579, MAE=2.4106
Seed 31415, RandomForest: R²=0.7123, RMSE=3.1998, MAE=2.3443
---


In [9]:

import pandas as pd

# Save Part 1 results
df_part1 = pd.DataFrame(results_part1)
df_part1.to_csv('uniform_split_results.csv', index=False)

# Compute mean±std for Part 1
summary_part1 = df_part1.groupby('model').agg({
    'r2': ['mean', 'std'],
    'rmse': ['mean', 'std'],
    'mae': ['mean', 'std']
}).round(4)
print("=== Part 1: Full Data Uniform Split Summary ===")
print(summary_part1)
print()

# Part 2: 2022+ subset
mask_2022plus = years >= 2022
X_post2022 = X[mask_2022plus]
y_post2022 = y[mask_2022plus]
years_post2022 = years[mask_2022plus]
print(f"2022+ subset: {X_post2022.shape}, years: {sorted(set(years_post2022))}")
print(f"Year distribution in 2022+: {pd.Series(years_post2022).value_counts().sort_index().to_dict()}")


=== Part 1: Full Data Uniform Split Summary ===
                  r2            rmse             mae        
                mean     std    mean     std    mean     std
model                                                       
ExtraTrees    0.6650  0.0328  3.3634  0.1081  2.4637  0.0654
LightGBM      0.6951  0.0300  3.2084  0.0939  2.3515  0.0754
RandomForest  0.6847  0.0239  3.2644  0.0580  2.3676  0.0430

2022+ subset: (4500, 106), years: [np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Year distribution in 2022+: {2022: 962, 2023: 1185, 2024: 1567, 2025: 786}


In [10]:

# Part 2: 2022+ subset uniform split
results_part2 = []

for seed in seeds:
    tr_idx, te_idx = uniform_time_split(X_post2022, y_post2022, years_post2022, test_size=0.2, random_state=seed)
    X_train, X_test = X_post2022[tr_idx], X_post2022[te_idx]
    y_train, y_test = y_post2022[tr_idx], y_post2022[te_idx]
    
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)
    
    # LightGBM
    model = LGBMRegressor(n_estimators=500, max_depth=8, learning_rate=0.05, random_state=seed, verbosity=-1, n_jobs=4)
    r2, rmse, mae, y_pred = evaluate_model(model, X_train_s, y_train, X_test_s, y_test)
    results_part2.append({'seed': seed, 'model': 'LightGBM', 'r2': r2, 'rmse': rmse, 'mae': mae})
    print(f"Seed {seed}, LightGBM: R²={r2:.4f}, RMSE={rmse:.4f}, MAE={mae:.4f}")
    
    # ExtraTrees
    model = ExtraTreesRegressor(n_estimators=300, max_depth=15, random_state=seed, n_jobs=4)
    r2, rmse, mae, y_pred = evaluate_model(model, X_train_s, y_train, X_test_s, y_test)
    results_part2.append({'seed': seed, 'model': 'ExtraTrees', 'r2': r2, 'rmse': rmse, 'mae': mae})
    print(f"Seed {seed}, ExtraTrees: R²={r2:.4f}, RMSE={rmse:.4f}, MAE={mae:.4f}")
    
    # RandomForest
    model = RandomForestRegressor(n_estimators=300, max_depth=15, random_state=seed, n_jobs=4)
    r2, rmse, mae, y_pred = evaluate_model(model, X_train_s, y_train, X_test_s, y_test)
    results_part2.append({'seed': seed, 'model': 'RandomForest', 'r2': r2, 'rmse': rmse, 'mae': mae})
    print(f"Seed {seed}, RandomForest: R²={r2:.4f}, RMSE={rmse:.4f}, MAE={mae:.4f}")
    print("---")

df_part2 = pd.DataFrame(results_part2)
df_part2.to_csv('output/post2022_uniform_results.csv', index=False)

summary_part2 = df_part2.groupby('model').agg({
    'r2': ['mean', 'std'],
    'rmse': ['mean', 'std'],
    'mae': ['mean', 'std']
}).round(4)
print("=== Part 2: 2022+ Subset Uniform Split Summary ===")
print(summary_part2)


Seed 42, LightGBM: R²=0.6788, RMSE=3.2754, MAE=2.4238
Seed 42, ExtraTrees: R²=0.6386, RMSE=3.4740, MAE=2.5367
Seed 42, RandomForest: R²=0.6715, RMSE=3.3121, MAE=2.4172
---
Seed 2024, LightGBM: R²=0.6768, RMSE=3.2486, MAE=2.3572
Seed 2024, ExtraTrees: R²=0.6546, RMSE=3.3583, MAE=2.4438
Seed 2024, RandomForest: R²=0.6703, RMSE=3.2812, MAE=2.3412
---
Seed 31415, LightGBM: R²=0.7298, RMSE=3.1011, MAE=2.2734
Seed 31415, ExtraTrees: R²=0.7018, RMSE=3.2579, MAE=2.4106
Seed 31415, RandomForest: R²=0.7123, RMSE=3.1998, MAE=2.3443
---
=== Part 2: 2022+ Subset Uniform Split Summary ===
                  r2            rmse             mae        
                mean     std    mean     std    mean     std
model                                                       
ExtraTrees    0.6650  0.0328  3.3634  0.1081  2.4637  0.0654
LightGBM      0.6951  0.0300  3.2084  0.0939  2.3515  0.0754
RandomForest  0.6847  0.0239  3.2644  0.0580  2.3676  0.0430


### 五折交叉验证（与 `uniform_time_split` 同年分层原则）

在每个 `pub_year` 内将样本随机打乱后，按 **round-robin** 划入 5 折；第 *k* 折在其余折上训练、在第 *k* 折上测试。若某年样本数 **小于 5**，该年样本标记为不参与验证（仅进入训练集），与单划分中小年全部进训练集的做法一致。

表格特征五折中拟合 **LightGBM、ExtraTrees、RandomForest、XGBoost、CatBoost**（需已安装 `xgboost`、`catboost`）。时序嵌入的五折见后文 Part 3 专用单元。


In [11]:
# 五折交叉验证（依赖 uniform_time_kfold_assign / iter_uniform_time_kfold 与 evaluate_model）
N_FOLDS = 5

from xgboost import XGBRegressor
from catboost import CatBoostRegressor


def run_uniform_time_5cv(X_arr, y_arr, years_arr, dataset_label, model_seed=42):
    rows = []
    for fold_id, tr_idx, te_idx in iter_uniform_time_kfold(years_arr, n_splits=N_FOLDS, random_state=42):
        if len(te_idx) == 0:
            print(f"[{dataset_label}] fold {fold_id}: 测试集为空，跳过")
            continue
        X_train, X_test = X_arr[tr_idx], X_arr[te_idx]
        y_train, y_test = y_arr[tr_idx], y_arr[te_idx]
        scaler = StandardScaler()
        X_train_s = scaler.fit_transform(X_train)
        X_test_s = scaler.transform(X_test)

        model = LGBMRegressor(
            n_estimators=500, max_depth=8, learning_rate=0.05,
            random_state=model_seed, verbosity=-1, n_jobs=4,
        )
        r2, rmse, mae, _ = evaluate_model(model, X_train_s, y_train, X_test_s, y_test)
        rows.append({"dataset": dataset_label, "fold": fold_id, "model": "LightGBM", "r2": r2, "rmse": rmse, "mae": mae})

        model = ExtraTreesRegressor(n_estimators=300, max_depth=15, random_state=model_seed, n_jobs=4)
        r2, rmse, mae, _ = evaluate_model(model, X_train_s, y_train, X_test_s, y_test)
        rows.append({"dataset": dataset_label, "fold": fold_id, "model": "ExtraTrees", "r2": r2, "rmse": rmse, "mae": mae})

        model = RandomForestRegressor(n_estimators=300, max_depth=15, random_state=model_seed, n_jobs=4)
        r2, rmse, mae, _ = evaluate_model(model, X_train_s, y_train, X_test_s, y_test)
        rows.append({"dataset": dataset_label, "fold": fold_id, "model": "RandomForest", "r2": r2, "rmse": rmse, "mae": mae})

        model = XGBRegressor(
            n_estimators=500,
            max_depth=8,
            learning_rate=0.05,
            random_state=model_seed,
            n_jobs=4,
            tree_method="hist",
        )
        r2, rmse, mae, _ = evaluate_model(model, X_train_s, y_train, X_test_s, y_test)
        rows.append({"dataset": dataset_label, "fold": fold_id, "model": "XGBoost", "r2": r2, "rmse": rmse, "mae": mae})

        model = CatBoostRegressor(
            iterations=500,
            depth=8,
            learning_rate=0.05,
            random_state=model_seed,
            verbose=False,
            loss_function="RMSE",
        )
        r2, rmse, mae, _ = evaluate_model(model, X_train_s, y_train, X_test_s, y_test)
        rows.append({"dataset": dataset_label, "fold": fold_id, "model": "CatBoost", "r2": r2, "rmse": rmse, "mae": mae})

        print(f"[{dataset_label}] fold {fold_id}: train={len(tr_idx)}, test={len(te_idx)}")
    return pd.DataFrame(rows)


import pandas as pd

parts = [run_uniform_time_5cv(X, y, years, "full_data")]
if "X_post2022" in globals() and "years_post2022" in globals():
    parts.append(run_uniform_time_5cv(X_post2022, y_post2022, years_post2022, "post_2022"))
else:
    print("提示: 未找到 X_post2022 / years_post2022，跳过 2022+ 子集五折（请先运行定义 Part2 子集的单元）。")

df_5cv = pd.concat(parts, ignore_index=True)
df_5cv.to_csv("uniform_time_5cv_results.csv", index=False)
print("Saved: uniform_time_5cv_results.csv\n")
print("=== 五折: 各模型 mean ± std ===")
print(df_5cv.groupby(["dataset", "model"])[["r2", "rmse", "mae"]].agg(["mean", "std"]).round(4))


[full_data] fold 0: train=3598, test=902
[full_data] fold 1: train=3599, test=901
[full_data] fold 2: train=3601, test=899
[full_data] fold 3: train=3601, test=899
[full_data] fold 4: train=3601, test=899
[post_2022] fold 0: train=3598, test=902
[post_2022] fold 1: train=3599, test=901
[post_2022] fold 2: train=3601, test=899
[post_2022] fold 3: train=3601, test=899
[post_2022] fold 4: train=3601, test=899
Saved: uniform_time_5cv_results.csv

=== 五折: 各模型 mean ± std ===
                            r2            rmse             mae        
                          mean     std    mean     std    mean     std
dataset   model                                                       
full_data ExtraTrees    0.6704  0.0316  3.3419  0.1953  2.4405  0.1193
          LightGBM      0.6910  0.0227  3.2369  0.1593  2.3430  0.0920
          RandomForest  0.6871  0.0281  3.2570  0.1843  2.3444  0.0999
post_2022 ExtraTrees    0.6704  0.0316  3.3419  0.1953  2.4405  0.1193
          LightGBM      0.691

In [12]:

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Sort data by (year, PCE)
df_sorted = df.sort_values(['pub_year', 'PCE']).reset_index(drop=True)
y_sorted = df_sorted['PCE'].values
years_sorted = df_sorted['pub_year'].values.astype(int)
X_sorted = df_sorted[feature_cols].fillna(0).values

from sklearn.preprocessing import StandardScaler
seq_feature_scaler = StandardScaler()
X_sorted_seq = seq_feature_scaler.fit_transform(X_sorted)
X_sorted_seq = np.nan_to_num(X_sorted_seq, nan=0.0, posinf=0.0, neginf=0.0)
print("Seq MAE: 106 维已 StandardScaler（按列 z-score），避免未缩放 MSE 数量级爆炸")

# Group by year（序列 MAE 用标准化特征；下游 tabular 仍用 X_sorted 原始值）
year_groups = {}
for year in sorted(np.unique(years_sorted)):
    mask = years_sorted == year
    indices = np.where(mask)[0]
    year_groups[year] = {
        'X': X_sorted_seq[mask],
        'y': y_sorted[mask],
        'indices': indices,
        'n': len(indices)
    }
    print(f"Year {year}: {len(indices)} samples")

max_year_len = max(g['n'] for g in year_groups.values())
print(f"\nMax year length: {max_year_len}")


Seq MAE: 106 维已 StandardScaler（按列 z-score），避免未缩放 MSE 数量级爆炸
Year 2022: 962 samples
Year 2023: 1185 samples
Year 2024: 1567 samples
Year 2025: 786 samples

Max year length: 1567


In [13]:

import torch
from torch.utils.data import Dataset, DataLoader

class YearChunkDataset(Dataset):
    def __init__(self, year_groups, max_len=256):
        self.max_len = max_len
        self.chunks = []  # list of (X_chunk, y_chunk)
        for year, group in year_groups.items():
            X = group['X']
            y = group['y']
            n = group['n']
            # Split into chunks
            for i in range(0, n, max_len):
                end = min(i + max_len, n)
                X_chunk = X[i:end]
                y_chunk = y[i:end]
                self.chunks.append((X_chunk, y_chunk, year, i))
    
    def __len__(self):
        return len(self.chunks)
    
    def __getitem__(self, idx):
        X_chunk, y_chunk, year, start_idx = self.chunks[idx]
        # Pad to max_len
        seq_len = len(X_chunk)
        if seq_len < self.max_len:
            pad = np.zeros((self.max_len - seq_len, X_chunk.shape[1]))
            X_chunk = np.vstack([X_chunk, pad])
            y_pad = np.zeros(self.max_len - seq_len)
            y_chunk = np.concatenate([y_chunk, y_pad])
        return torch.FloatTensor(X_chunk), torch.FloatTensor(y_chunk), torch.tensor(seq_len), torch.tensor(year), torch.tensor(start_idx)

# Create dataset
dataset = YearChunkDataset(year_groups, max_len=256)
print(f"Total chunks: {len(dataset)}")

# Dataloader with custom collate
def collate_fn(batch):
    X = torch.stack([b[0] for b in batch])  # [B, T, F]
    y = torch.stack([b[1] for b in batch])  # [B, T]
    lens = torch.stack([b[2] for b in batch])  # [B]
    years = torch.stack([b[3] for b in batch])
    starts = torch.stack([b[4] for b in batch])
    return X, y, lens, years, starts

# For simplicity, batch_size=1 to avoid padding issues between different year chunks
# Actually we can use batch_size=2 since all are padded to 256
dataloader = DataLoader(dataset, batch_size=2, shuffle=True, collate_fn=collate_fn)

# Test one batch
for batch in dataloader:
    X_b, y_b, lens, years_b, starts = batch
    print(f"X shape: {X_b.shape}, y shape: {y_b.shape}, lens: {lens}")
    break


Total chunks: 20
X shape: torch.Size([2, 256, 106]), y shape: torch.Size([2, 256]), lens: tensor([256, 256])


In [14]:

# 序列编码器：Masked Autoencoding（无 PCE 标签）。用 recon_head 重建被遮挡输入特征，避免 pred_head 监督 PCE 导致的隐式标签泄露。
class LSTMFeatureExtractor(nn.Module):
    def __init__(self, input_dim=106, hidden_dim=128, num_layers=2):
        super().__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        drop = 0.1 if num_layers > 1 else 0.0
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, dropout=drop)
        self.recon_head = nn.Linear(hidden_dim, input_dim)

    def forward(self, x, seq_lens):
        packed = nn.utils.rnn.pack_padded_sequence(
            x, seq_lens.cpu(), batch_first=True, enforce_sorted=False
        )
        packed_out, _ = self.lstm(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(
            packed_out, batch_first=True, total_length=x.size(1)
        )
        recon = self.recon_head(out)
        return recon, out

class GRUFeatureExtractor(nn.Module):
    def __init__(self, input_dim=106, hidden_dim=128, num_layers=2):
        super().__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        drop = 0.1 if num_layers > 1 else 0.0
        self.gru = nn.GRU(input_dim, hidden_dim, num_layers, batch_first=True, dropout=drop)
        self.recon_head = nn.Linear(hidden_dim, input_dim)

    def forward(self, x, seq_lens):
        packed = nn.utils.rnn.pack_padded_sequence(
            x, seq_lens.cpu(), batch_first=True, enforce_sorted=False
        )
        packed_out, _ = self.gru(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(
            packed_out, batch_first=True, total_length=x.size(1)
        )
        recon = self.recon_head(out)
        return recon, out

class TransformerFeatureExtractor(nn.Module):
    def __init__(self, input_dim=106, d_model=128, nhead=4, num_layers=2):
        super().__init__()
        self.input_dim = input_dim
        self.input_proj = nn.Linear(input_dim, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=256, dropout=0.1,
            batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.recon_head = nn.Linear(d_model, input_dim)

    def forward(self, x, seq_lens):
        B, T, _ = x.shape
        x_proj = self.input_proj(x)
        mask = torch.zeros(B, T, dtype=torch.bool, device=x.device)
        for i, l in enumerate(seq_lens):
            mask[i, l:] = True
        out = self.transformer(x_proj, src_key_padding_mask=mask)
        recon = self.recon_head(out)
        return recon, out

print("Models defined successfully (MAE / no PCE head)")


Models defined successfully (MAE / no PCE head)


In [15]:

def train_mae_pretrain(model, dataloader, epochs=100, lr=0.0005, device='cpu', mask_ratio=0.2,
                        max_grad_norm=1.0):
    """年度序列：随机 mask 特征维度并重建；输入应为标准化特征；不使用 PCE。"""
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        n_batches = 0
        for X_b, y_b, lens, _, _ in dataloader:
            X_b = X_b.to(device)
            lens = lens.to(device)
            B, T, fdim = X_b.shape
            valid = torch.arange(T, device=device).unsqueeze(0) < lens.unsqueeze(1)
            feat_mask = torch.rand(B, T, fdim, device=device) < mask_ratio
            mask = feat_mask & valid.unsqueeze(-1)
            if mask.sum().item() == 0:
                continue
            x_in = X_b.clone()
            x_in[mask] = 0.0
            recon, _ = model(x_in, lens)
            loss = torch.nn.functional.smooth_l1_loss(recon[mask], X_b[mask], reduction='mean')
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            optimizer.step()
            total_loss += loss.item()
            n_batches += 1
        denom = max(n_batches, 1)
        print(f"  Epoch {epoch+1}/{epochs}, Loss: {total_loss/denom:.4f}")
    return model


def freeze_encoder(model):
    for p in model.parameters():
        p.requires_grad = False


def extract_features(model, dataset, device='cpu'):
    """按样本聚合 hidden state；Encoder 应在下游任务前冻结。"""
    model.to(device)
    model.eval()
    feature_map = {}
    dataloader_inf = DataLoader(dataset, batch_size=4, shuffle=False, collate_fn=collate_fn)
    with torch.no_grad():
        for X_b, y_b, lens, years_b, starts in dataloader_inf:
            X_b = X_b.to(device)
            lens = lens.to(device)
            _, hidden = model(X_b, lens)
            for i in range(len(lens)):
                year = years_b[i].item()
                start = starts[i].item()
                l = lens[i].item()
                for j in range(l):
                    global_idx = year_groups[year]['indices'][start + j]
                    feature_map[global_idx] = hidden[i, j].cpu().numpy()
    n_samples = len(df_sorted)
    hidden_dim = list(feature_map.values())[0].shape[0]
    features = np.zeros((n_samples, hidden_dim))
    for idx in range(n_samples):
        features[idx] = feature_map[idx]
    return features


print("=== MAE pretrain: LSTM ===")
lstm_model = LSTMFeatureExtractor(input_dim=106, hidden_dim=128, num_layers=2)
lstm_model = train_mae_pretrain(lstm_model, dataloader, epochs=100, lr=0.0005)
freeze_encoder(lstm_model)
print("Extracting LSTM features...")
lstm_features = extract_features(lstm_model, dataset)
print(f"LSTM features shape: {lstm_features.shape}")


=== MAE pretrain: LSTM ===
  Epoch 1/100, Loss: 0.0663
  Epoch 2/100, Loss: 0.0630
  Epoch 3/100, Loss: 0.0651
  Epoch 4/100, Loss: 0.0595
  Epoch 5/100, Loss: 0.0623
  Epoch 6/100, Loss: 0.0624
  Epoch 7/100, Loss: 0.0621
  Epoch 8/100, Loss: 0.0596
  Epoch 9/100, Loss: 0.0577
  Epoch 10/100, Loss: 0.0573
  Epoch 11/100, Loss: 0.0517
  Epoch 12/100, Loss: 0.0502
  Epoch 13/100, Loss: 0.0467
  Epoch 14/100, Loss: 0.0466
  Epoch 15/100, Loss: 0.0474
  Epoch 16/100, Loss: 0.0437
  Epoch 17/100, Loss: 0.0453
  Epoch 18/100, Loss: 0.0468
  Epoch 19/100, Loss: 0.0427
  Epoch 20/100, Loss: 0.0420
  Epoch 21/100, Loss: 0.0405
  Epoch 22/100, Loss: 0.0418
  Epoch 23/100, Loss: 0.0388
  Epoch 24/100, Loss: 0.0402
  Epoch 25/100, Loss: 0.0429
  Epoch 26/100, Loss: 0.0407
  Epoch 27/100, Loss: 0.0432
  Epoch 28/100, Loss: 0.0394
  Epoch 29/100, Loss: 0.0396
  Epoch 30/100, Loss: 0.0387
  Epoch 31/100, Loss: 0.0372
  Epoch 32/100, Loss: 0.0381
  Epoch 33/100, Loss: 0.0376
  Epoch 34/100, Loss: 0.0

In [16]:
torch.save(lstm_model.state_dict(), 'lstm_model.pth')

In [17]:
# Train Transformer（MAE；先 freeze_encoder 再提取 embedding）
trans_model = TransformerFeatureExtractor(input_dim=106, d_model=128, nhead=8, num_layers=8)
trans_model = train_mae_pretrain(trans_model, dataloader, epochs=300, lr=0.0001)
freeze_encoder(trans_model)
print("Transformer training done, extracting features...")
trans_features = extract_features(trans_model, dataset)
print(f"Transformer features: {trans_features.shape}")

torch.save(trans_model.state_dict(), 'trans_model.pth')


  Epoch 1/300, Loss: 0.1860
  Epoch 2/300, Loss: 0.1068
  Epoch 3/300, Loss: 0.0869
  Epoch 4/300, Loss: 0.0735
  Epoch 5/300, Loss: 0.0716
  Epoch 6/300, Loss: 0.0684
  Epoch 7/300, Loss: 0.0736
  Epoch 8/300, Loss: 0.0694
  Epoch 9/300, Loss: 0.0697
  Epoch 10/300, Loss: 0.0654
  Epoch 11/300, Loss: 0.0644
  Epoch 12/300, Loss: 0.0676
  Epoch 13/300, Loss: 0.0663
  Epoch 14/300, Loss: 0.0656
  Epoch 15/300, Loss: 0.0668
  Epoch 16/300, Loss: 0.0613
  Epoch 17/300, Loss: 0.0662
  Epoch 18/300, Loss: 0.0599
  Epoch 19/300, Loss: 0.0597
  Epoch 20/300, Loss: 0.0578
  Epoch 21/300, Loss: 0.0609
  Epoch 22/300, Loss: 0.0595
  Epoch 23/300, Loss: 0.0594
  Epoch 24/300, Loss: 0.0568
  Epoch 25/300, Loss: 0.0540
  Epoch 26/300, Loss: 0.0550
  Epoch 27/300, Loss: 0.0550
  Epoch 28/300, Loss: 0.0598
  Epoch 29/300, Loss: 0.0526
  Epoch 30/300, Loss: 0.0508
  Epoch 31/300, Loss: 0.0514
  Epoch 32/300, Loss: 0.0458
  Epoch 33/300, Loss: 0.0485
  Epoch 34/300, Loss: 0.0527
  Epoch 35/300, Loss: 0

In [18]:

# Train GRU (MAE)
gru_model = GRUFeatureExtractor(input_dim=106, hidden_dim=128, num_layers=2)
gru_model = train_mae_pretrain(gru_model, dataloader, epochs=100, lr=0.0005)
freeze_encoder(gru_model)
print("GRU training done, extracting features...")
gru_features = extract_features(gru_model, dataset)
print(f"GRU features: {gru_features.shape}")

torch.save(gru_model.state_dict(), 'gru_model.pth')

  Epoch 1/100, Loss: 0.0664
  Epoch 2/100, Loss: 0.0651
  Epoch 3/100, Loss: 0.0634
  Epoch 4/100, Loss: 0.0624
  Epoch 5/100, Loss: 0.0641
  Epoch 6/100, Loss: 0.0586
  Epoch 7/100, Loss: 0.0542
  Epoch 8/100, Loss: 0.0535
  Epoch 9/100, Loss: 0.0504
  Epoch 10/100, Loss: 0.0499
  Epoch 11/100, Loss: 0.0474
  Epoch 12/100, Loss: 0.0470
  Epoch 13/100, Loss: 0.0477
  Epoch 14/100, Loss: 0.0408
  Epoch 15/100, Loss: 0.0395
  Epoch 16/100, Loss: 0.0448
  Epoch 17/100, Loss: 0.0393
  Epoch 18/100, Loss: 0.0357
  Epoch 19/100, Loss: 0.0385
  Epoch 20/100, Loss: 0.0414
  Epoch 21/100, Loss: 0.0382
  Epoch 22/100, Loss: 0.0374
  Epoch 23/100, Loss: 0.0365
  Epoch 24/100, Loss: 0.0376
  Epoch 25/100, Loss: 0.0364
  Epoch 26/100, Loss: 0.0383
  Epoch 27/100, Loss: 0.0343
  Epoch 28/100, Loss: 0.0359
  Epoch 29/100, Loss: 0.0363
  Epoch 30/100, Loss: 0.0314
  Epoch 31/100, Loss: 0.0389
  Epoch 32/100, Loss: 0.0346
  Epoch 33/100, Loss: 0.0336
  Epoch 34/100, Loss: 0.0332
  Epoch 35/100, Loss: 0

In [19]:

# Work with sorted data for Part 3
X_sorted_numeric = df_sorted[feature_cols].fillna(0).values  # [7266, 106]
y_sorted_arr = df_sorted['PCE'].values
years_sorted_arr = df_sorted['pub_year'].values.astype(int)

print(f"Sorted data: X={X_sorted_numeric.shape}, y={y_sorted_arr.shape}")
print(f"Temporal features: LSTM={lstm_features.shape}, GRU={gru_features.shape}, Transformer={trans_features.shape}")

# 与 2.ipynb 对齐：排序后第 j 行对应原 df 的 index（CSV 行标识）
df_clean = df.dropna(subset=['PCE'])
sort_ix = np.lexsort((df_clean['PCE'].values, df_clean['pub_year'].values))
sorted_source_index = df_clean.index.values[sort_ix].astype(np.int64)

# Save sequence data
np.savez('output/yearly_sequence_data.npz',
         X_numeric=X_sorted_numeric,
         y=y_sorted_arr,
         years=years_sorted_arr,
         lstm_features=lstm_features,
         gru_features=gru_features,
         transformer_features=trans_features,
         sorted_source_index=sorted_source_index)
print("Saved yearly_sequence_data.npz（含 sorted_source_index）")

# Define feature combinations
feature_sets = {
    'original_106d': X_sorted_numeric,
    'original_LSTM_234d': np.hstack([X_sorted_numeric, lstm_features]),
    'original_Transformer_234d': np.hstack([X_sorted_numeric, trans_features]),
    'original_GRU_234d': np.hstack([X_sorted_numeric, gru_features]),
    'original_all_490d': np.hstack([X_sorted_numeric, lstm_features, trans_features, gru_features]),
}
for name, feats in feature_sets.items():
    print(f"{name}: {feats.shape}")

# 供 2.ipynb：490d 集成教师 LGBM + ExtraTrees（与 run_full_pipeline 一致，默认 0.5/0.5）
import joblib
from sklearn.preprocessing import StandardScaler as _SS490
from lightgbm import LGBMRegressor as _LGBM
from sklearn.ensemble import ExtraTreesRegressor as _ET
from sklearn.metrics import r2_score as _r2, mean_squared_error as _mse
X_490 = feature_sets['original_all_490d']
_seed = 42
_tr, _te = uniform_time_split(X_sorted_numeric, y_sorted_arr, years_sorted_arr, test_size=0.2, random_state=_seed)
_sc490 = _SS490()
_Xtr = _sc490.fit_transform(X_490[_tr])
_Xte = _sc490.transform(X_490[_te])
_lgb = _LGBM(n_estimators=500, max_depth=8, learning_rate=0.05, random_state=_seed, verbosity=-1, n_jobs=4)
_lgb.fit(_Xtr, y_sorted_arr[_tr])
_pred = _lgb.predict(_Xte)
_r2_lgb = float(_r2(y_sorted_arr[_te], _pred))
_rms_lgb = float(np.sqrt(_mse(y_sorted_arr[_te], _pred)))
_et = _ET(n_estimators=300, max_depth=15, random_state=_seed, n_jobs=4)
_et.fit(_Xtr, y_sorted_arr[_tr])
_pred_et = _et.predict(_Xte)
_r2_et = float(_r2(y_sorted_arr[_te], _pred_et))
_rms_et = float(np.sqrt(_mse(y_sorted_arr[_te], _pred_et)))
_wl, _we = 0.5, 0.5
_pred_ens = _wl * _pred + _we * _pred_et
_r2s = float(_r2(y_sorted_arr[_te], _pred_ens))
_rms = float(np.sqrt(_mse(y_sorted_arr[_te], _pred_ens)))
print(f"[export] 490d LGBM R²={_r2_lgb:.4f} | ET R²={_r2_et:.4f} | 加权 R²={_r2s:.4f} RMSE={_rms:.4f}")
joblib.dump({
    'lgbm': _lgb, 'extratrees': _et, 'scaler_490': _sc490,
    'teacher_w_lgbm': _wl, 'teacher_w_et': _we,
    'tr_idx': _tr, 'te_idx': _te,
    'r2_test_lgbm': _r2_lgb, 'rmse_test_lgbm': _rms_lgb,
    'r2_test_extratrees': _r2_et, 'rmse_test_extratrees': _rms_et,
    'r2_test_ensemble': _r2s, 'rmse_test_ensemble': _rms,
    'r2_test': _r2s, 'rmse_test': _rms,
}, 'output/pce_lgbm_490d_bundle.joblib')
print("Saved output/pce_lgbm_490d_bundle.joblib")


Sorted data: X=(4500, 106), y=(4500,)
Temporal features: LSTM=(4500, 128), GRU=(4500, 128), Transformer=(4500, 128)
Saved yearly_sequence_data.npz（含 sorted_source_index）
original_106d: (4500, 106)
original_LSTM_234d: (4500, 234)
original_Transformer_234d: (4500, 234)
original_GRU_234d: (4500, 234)
original_all_490d: (4500, 490)
[export] 490d LGBM R²=0.8781 | ET R²=0.8474 | 加权 R²=0.8718 RMSE=2.0146
Saved output/pce_lgbm_490d_bundle.joblib


In [20]:

# Part 3: Tree models on temporal feature combinations
results_part3 = []

seed = 42
tr_idx, te_idx = uniform_time_split(X_sorted_numeric, y_sorted_arr, years_sorted_arr, test_size=0.2, random_state=seed)

for feat_name, X_feat in feature_sets.items():
    X_train, X_test = X_feat[tr_idx], X_feat[te_idx]
    y_train, y_test = y_sorted_arr[tr_idx], y_sorted_arr[te_idx]
    
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)
    
    # LightGBM
    model = LGBMRegressor(n_estimators=500, max_depth=8, learning_rate=0.05, random_state=seed, verbosity=-1, n_jobs=4)
    r2, rmse, mae, y_pred = evaluate_model(model, X_train_s, y_train, X_test_s, y_test)
    results_part3.append({'feature_set': feat_name, 'model': 'LightGBM', 'r2': r2, 'rmse': rmse, 'mae': mae})
    print(f"{feat_name} + LightGBM: R²={r2:.4f}")
    
    # ExtraTrees
    model = ExtraTreesRegressor(n_estimators=300, max_depth=15, random_state=seed, n_jobs=4)
    r2, rmse, mae, y_pred = evaluate_model(model, X_train_s, y_train, X_test_s, y_test)
    results_part3.append({'feature_set': feat_name, 'model': 'ExtraTrees', 'r2': r2, 'rmse': rmse, 'mae': mae})
    print(f"{feat_name} + ExtraTrees: R²={r2:.4f}")
    print("---")

print("\n=== Part 3 Results ===")
for r in results_part3:
    print(f"{r['feature_set']} + {r['model']}: R²={r['r2']:.4f}, RMSE={r['rmse']:.4f}, MAE={r['mae']:.4f}")


original_106d + LightGBM: R²=0.6532
original_106d + ExtraTrees: R²=0.6227
---
original_LSTM_234d + LightGBM: R²=0.7620
original_LSTM_234d + ExtraTrees: R²=0.7404
---
original_Transformer_234d + LightGBM: R²=0.8668
original_Transformer_234d + ExtraTrees: R²=0.8474
---
original_GRU_234d + LightGBM: R²=0.7639
original_GRU_234d + ExtraTrees: R²=0.7479
---
original_all_490d + LightGBM: R²=0.8781
original_all_490d + ExtraTrees: R²=0.8474
---

=== Part 3 Results ===
original_106d + LightGBM: R²=0.6532, RMSE=3.3140, MAE=2.3931
original_106d + ExtraTrees: R²=0.6227, RMSE=3.4566, MAE=2.5275
original_LSTM_234d + LightGBM: R²=0.7620, RMSE=2.7452, MAE=1.9944
original_LSTM_234d + ExtraTrees: R²=0.7404, RMSE=2.8672, MAE=2.1027
original_Transformer_234d + LightGBM: R²=0.8668, RMSE=2.0541, MAE=1.3980
original_Transformer_234d + ExtraTrees: R²=0.8474, RMSE=2.1983, MAE=1.5200
original_GRU_234d + LightGBM: R²=0.7639, RMSE=2.7345, MAE=1.9811
original_GRU_234d + ExtraTrees: R²=0.7479, RMSE=2.8255, MAE=2.052

In [21]:

# Faster models for Part 3
results_part3 = []
seed = 42
tr_idx, te_idx = uniform_time_split(X_sorted_numeric, y_sorted_arr, years_sorted_arr, test_size=0.2, random_state=seed)

# Process original baseline first
X_train, X_test = X_sorted_numeric[tr_idx], X_sorted_numeric[te_idx]
y_train, y_test = y_sorted_arr[tr_idx], y_sorted_arr[te_idx]
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

model = LGBMRegressor(n_estimators=200, max_depth=8, learning_rate=0.05, random_state=seed, verbosity=-1, n_jobs=4)
r2, rmse, mae, _ = evaluate_model(model, X_train_s, y_train, X_test_s, y_test)
results_part3.append({'feature_set': 'original_106d', 'model': 'LightGBM', 'r2': r2, 'rmse': rmse, 'mae': mae})
print(f"original_106d + LightGBM: R²={r2:.4f}")

model = ExtraTreesRegressor(n_estimators=150, max_depth=15, random_state=seed, n_jobs=4)
r2, rmse, mae, _ = evaluate_model(model, X_train_s, y_train, X_test_s, y_test)
results_part3.append({'feature_set': 'original_106d', 'model': 'ExtraTrees', 'r2': r2, 'rmse': rmse, 'mae': mae})
print(f"original_106d + ExtraTrees: R²={r2:.4f}")


original_106d + LightGBM: R²=0.6323
original_106d + ExtraTrees: R²=0.6203


In [22]:

# LSTM concat
X_lstm = np.hstack([X_sorted_numeric, lstm_features])
X_train, X_test = X_lstm[tr_idx], X_lstm[te_idx]
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

model = LGBMRegressor(n_estimators=200, max_depth=8, learning_rate=0.05, random_state=seed, verbosity=-1, n_jobs=4)
r2, rmse, mae, _ = evaluate_model(model, X_train_s, y_train, X_test_s, y_test)
results_part3.append({'feature_set': 'original_LSTM_234d', 'model': 'LightGBM', 'r2': r2, 'rmse': rmse, 'mae': mae})
print(f"original+LSTM + LightGBM: R²={r2:.4f}")

model = ExtraTreesRegressor(n_estimators=150, max_depth=15, random_state=seed, n_jobs=4)
r2, rmse, mae, _ = evaluate_model(model, X_train_s, y_train, X_test_s, y_test)
results_part3.append({'feature_set': 'original_LSTM_234d', 'model': 'ExtraTrees', 'r2': r2, 'rmse': rmse, 'mae': mae})
print(f"original+LSTM + ExtraTrees: R²={r2:.4f}")


original+LSTM + LightGBM: R²=0.7543
original+LSTM + ExtraTrees: R²=0.7403


In [23]:

# Transformer concat
X_trans = np.hstack([X_sorted_numeric, trans_features])
X_train, X_test = X_trans[tr_idx], X_trans[te_idx]
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

model = LGBMRegressor(n_estimators=200, max_depth=8, learning_rate=0.05, random_state=seed, verbosity=-1, n_jobs=4)
r2, rmse, mae, _ = evaluate_model(model, X_train_s, y_train, X_test_s, y_test)
results_part3.append({'feature_set': 'original_Transformer_234d', 'model': 'LightGBM', 'r2': r2, 'rmse': rmse, 'mae': mae})
print(f"original+Transformer + LightGBM: R²={r2:.4f}")

model = ExtraTreesRegressor(n_estimators=150, max_depth=15, random_state=seed, n_jobs=4)
r2, rmse, mae, _ = evaluate_model(model, X_train_s, y_train, X_test_s, y_test)
results_part3.append({'feature_set': 'original_Transformer_234d', 'model': 'ExtraTrees', 'r2': r2, 'rmse': rmse, 'mae': mae})
print(f"original+Transformer + ExtraTrees: R²={r2:.4f}")

# GRU concat
X_gru = np.hstack([X_sorted_numeric, gru_features])
X_train, X_test = X_gru[tr_idx], X_gru[te_idx]
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

model = LGBMRegressor(n_estimators=200, max_depth=8, learning_rate=0.05, random_state=seed, verbosity=-1, n_jobs=4)
r2, rmse, mae, _ = evaluate_model(model, X_train_s, y_train, X_test_s, y_test)
results_part3.append({'feature_set': 'original_GRU_234d', 'model': 'LightGBM', 'r2': r2, 'rmse': rmse, 'mae': mae})
print(f"original+GRU + LightGBM: R²={r2:.4f}")

model = ExtraTreesRegressor(n_estimators=150, max_depth=15, random_state=seed, n_jobs=4)
r2, rmse, mae, _ = evaluate_model(model, X_train_s, y_train, X_test_s, y_test)
results_part3.append({'feature_set': 'original_GRU_234d', 'model': 'ExtraTrees', 'r2': r2, 'rmse': rmse, 'mae': mae})
print(f"original+GRU + ExtraTrees: R²={r2:.4f}")


original+Transformer + LightGBM: R²=0.8599
original+Transformer + ExtraTrees: R²=0.8456
original+GRU + LightGBM: R²=0.7623
original+GRU + ExtraTrees: R²=0.7483


In [24]:

# All temporal features combined
X_all_temp = np.hstack([X_sorted_numeric, lstm_features, trans_features, gru_features])
X_train, X_test = X_all_temp[tr_idx], X_all_temp[te_idx]
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

model = LGBMRegressor(n_estimators=200, max_depth=8, learning_rate=0.05, random_state=seed, verbosity=-1, n_jobs=4)
r2, rmse, mae, _ = evaluate_model(model, X_train_s, y_train, X_test_s, y_test)
results_part3.append({'feature_set': 'original_all_490d', 'model': 'LightGBM', 'r2': r2, 'rmse': rmse, 'mae': mae})
print(f"original+all_temporal + LightGBM: R²={r2:.4f}")

model = ExtraTreesRegressor(n_estimators=150, max_depth=15, random_state=seed, n_jobs=4)
r2, rmse, mae, _ = evaluate_model(model, X_train_s, y_train, X_test_s, y_test)
results_part3.append({'feature_set': 'original_all_490d', 'model': 'ExtraTrees', 'r2': r2, 'rmse': rmse, 'mae': mae})
print(f"original+all_temporal + ExtraTrees: R²={r2:.4f}")


original+all_temporal + LightGBM: R²=0.8711
original+all_temporal + ExtraTrees: R²=0.8469


In [25]:

# Step 5: Historical statistical features
# Compute yearly PCE statistics
yearly_stats = df_sorted.groupby('pub_year')['PCE'].agg(['mean', 'std', 'max', 'median']).reset_index()
yearly_stats.columns = ['pub_year', 'year_pce_mean', 'year_pce_std', 'year_pce_max', 'year_pce_median']

# Merge back to dataframe
df_with_stats = df_sorted.merge(yearly_stats, on='pub_year', how='left')
stat_features = df_with_stats[['year_pce_mean', 'year_pce_std', 'year_pce_max', 'year_pce_median']].values

# Concatenate with original features
X_stat = np.hstack([X_sorted_numeric, stat_features])
print(f"Original + statistical features: {X_stat.shape}")

# Test
X_train, X_test = X_stat[tr_idx], X_stat[te_idx]
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

model = LGBMRegressor(n_estimators=200, max_depth=8, learning_rate=0.05, random_state=seed, verbosity=-1, n_jobs=4)
r2, rmse, mae, _ = evaluate_model(model, X_train_s, y_train, X_test_s, y_test)
results_part3.append({'feature_set': 'original_stats_110d', 'model': 'LightGBM', 'r2': r2, 'rmse': rmse, 'mae': mae})
print(f"original+stats + LightGBM: R²={r2:.4f}")

model = ExtraTreesRegressor(n_estimators=150, max_depth=15, random_state=seed, n_jobs=4)
r2, rmse, mae, _ = evaluate_model(model, X_train_s, y_train, X_test_s, y_test)
results_part3.append({'feature_set': 'original_stats_110d', 'model': 'ExtraTrees', 'r2': r2, 'rmse': rmse, 'mae': mae})
print(f"original+stats + ExtraTrees: R²={r2:.4f}")

# Save Part 3 results
df_part3 = pd.DataFrame(results_part3)
df_part3.to_csv('output/temporal_feature_concat_results.csv', index=False)
print("\n=== Part 3 Complete ===")
print(df_part3.to_string())


Original + statistical features: (4500, 110)
original+stats + LightGBM: R²=0.6421
original+stats + ExtraTrees: R²=0.6258

=== Part 3 Complete ===
                  feature_set       model        r2      rmse       mae
0               original_106d    LightGBM  0.632297  3.412407  2.477102
1               original_106d  ExtraTrees  0.620276  3.467741  2.534006
2          original_LSTM_234d    LightGBM  0.754255  2.789681  2.031661
3          original_LSTM_234d  ExtraTrees  0.740346  2.867544  2.100609
4   original_Transformer_234d    LightGBM  0.859912  2.106263  1.449074
5   original_Transformer_234d  ExtraTrees  0.845590  2.211314  1.531315
6           original_GRU_234d    LightGBM  0.762320  2.743527  1.986464
7           original_GRU_234d  ExtraTrees  0.748337  2.823072  2.048337
8           original_all_490d    LightGBM  0.871059  2.020724  1.411874
9           original_all_490d  ExtraTrees  0.846906  2.201869  1.526886
10        original_stats_110d    LightGBM  0.642119  3.366524 

### Part 3 时序嵌入：五折交叉验证

在 **已训练 MAE 编码器并拼接好 `feature_sets`** 的前提下，对每个特征矩阵（106d / +LSTM / +Transformer / +GRU / 490d 等）用与上文相同的 **`iter_uniform_time_kfold(years_sorted_arr)`** 做五折；每折在训练折上 `StandardScaler` 后拟合 **LightGBM、ExtraTrees、RandomForest、XGBoost、CatBoost**。

说明：嵌入向量在整个排序表上 **一次性提取**（与当前 Part 3 脚本一致）；五折只重训 **树模型头**，不把 MAE 编码器按折重训。


In [ ]:
# Part 3 时序嵌入：五折 CV（依赖 N_FOLDS、iter_uniform_time_kfold、evaluate_model 及 Part 3 的 feature_sets）
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

if "feature_sets" not in globals() or "years_sorted_arr" not in globals() or "y_sorted_arr" not in globals():
    print("请先运行 Part 3 中构造 feature_sets、y_sorted_arr、years_sorted_arr 的单元。")
else:
    rows_te = []
    model_cfgs = [
        (
            "LightGBM",
            LGBMRegressor,
            dict(
                n_estimators=500,
                max_depth=8,
                learning_rate=0.05,
                random_state=42,
                verbosity=-1,
                n_jobs=4,
            ),
        ),
        (
            "ExtraTrees",
            ExtraTreesRegressor,
            dict(n_estimators=300, max_depth=15, random_state=42, n_jobs=4),
        ),
        (
            "RandomForest",
            RandomForestRegressor,
            dict(n_estimators=300, max_depth=15, random_state=42, n_jobs=4),
        ),
        (
            "XGBoost",
            XGBRegressor,
            dict(
                n_estimators=500,
                max_depth=8,
                learning_rate=0.05,
                random_state=42,
                n_jobs=4,
                tree_method="hist",
            ),
        ),
        (
            "CatBoost",
            CatBoostRegressor,
            dict(
                iterations=500,
                depth=8,
                learning_rate=0.05,
                random_state=42,
                verbose=False,
                loss_function="RMSE",
                allow_writing_files=False,
            ),
        ),
    ]

    for feat_name, X_feat in feature_sets.items():
        for fold_id, tr_idx, te_idx in iter_uniform_time_kfold(
            years_sorted_arr, n_splits=N_FOLDS, random_state=42
        ):
            if len(te_idx) == 0:
                continue
            X_train, X_test = X_feat[tr_idx], X_feat[te_idx]
            y_train, y_test = y_sorted_arr[tr_idx], y_sorted_arr[te_idx]
            scaler = StandardScaler()
            X_train_s = scaler.fit_transform(X_train)
            X_test_s = scaler.transform(X_test)
            for model_name, Model, kw in model_cfgs:
                m = Model(**kw)
                r2, rmse, mae, _ = evaluate_model(m, X_train_s, y_train, X_test_s, y_test)
                rows_te.append(
                    {
                        "feature_set": feat_name,
                        "fold": fold_id,
                        "model": model_name,
                        "r2": r2,
                        "rmse": rmse,
                        "mae": mae,
                    }
                )
        print(f"[temporal 5-CV] feature_set={feat_name} 完成")

    df_te_5cv = pd.DataFrame(rows_te)
    df_te_5cv.to_csv("uniform_time_5cv_temporal_embedding.csv", index=False)
    print("\nSaved: uniform_time_5cv_temporal_embedding.csv")
    print("=== 时序嵌入五折: mean ± std ===")
    print(
        df_te_5cv.groupby(["feature_set", "model"])[["r2", "rmse", "mae"]]
        .agg(["mean", "std"])
        .round(4)
    )


In [26]:

# Part 4: Simple temporal factor concatenation
# Build temporal factors for each sample

# 1. Year
year_factor = years_sorted_arr.reshape(-1, 1).astype(float)

# 2. Polynomial time
year_poly = np.hstack([year_factor, year_factor**2, year_factor**3])

# 3. Tech generation (define generations based on year ranges)
# Gen1: early research 2014-2017, Gen2: development 2018-2021, Gen3: recent 2022+
gen = np.zeros((len(years_sorted_arr), 3))
for i, yr in enumerate(years_sorted_arr):
    if yr <= 2017:
        gen[i, 0] = 1  # gen1
    elif yr <= 2021:
        gen[i, 1] = 1  # gen2
    else:
        gen[i, 2] = 1  # gen3

# 4. Historical sliding average PCE (2-year window)
# Compute year -> mean PCE
yearly_mean_pce = df_sorted.groupby('pub_year')['PCE'].mean().to_dict()
sorted_years_list = sorted(yearly_mean_pce.keys())
slide_avg = np.zeros((len(years_sorted_arr), 1))
for i, yr in enumerate(years_sorted_arr):
    # 2-year window: current year and previous year
    vals = []
    for dy in [0, -1]:
        y_check = yr + dy
        if y_check in yearly_mean_pce:
            vals.append(yearly_mean_pce[y_check])
    slide_avg[i, 0] = np.mean(vals) if vals else yearly_mean_pce.get(yr, 0)

# 5. Cumulative max PCE (up to and including current year)
cum_max = np.zeros((len(years_sorted_arr), 1))
max_so_far = 0
for yr in sorted_years_list:
    yr_pce_max = df_sorted[df_sorted['pub_year'] == yr]['PCE'].max()
    max_so_far = max(max_so_far, yr_pce_max)
    mask = years_sorted_arr == yr
    cum_max[mask, 0] = max_so_far

# 6. Time difference from breakthrough years
breakthrough_diff = np.zeros((len(years_sorted_arr), 2))
for i, yr in enumerate(years_sorted_arr):
    breakthrough_diff[i, 0] = abs(yr - 2016)
    breakthrough_diff[i, 1] = abs(yr - 2022)

# 7. Annual PCE growth rate (year-over-year % change in mean PCE)
growth_rate = np.zeros((len(years_sorted_arr), 1))
for i, yr in enumerate(years_sorted_arr):
    prev_yr = yr - 1
    if prev_yr in yearly_mean_pce and yearly_mean_pce[prev_yr] > 0:
        growth_rate[i, 0] = (yearly_mean_pce[yr] - yearly_mean_pce[prev_yr]) / yearly_mean_pce[prev_yr] * 100
    else:
        growth_rate[i, 0] = 0

print("Temporal factors created:")
print(f"  year: {year_factor.shape}")
print(f"  year_poly: {year_poly.shape}")
print(f"  gen: {gen.shape}")
print(f"  slide_avg: {slide_avg.shape}")
print(f"  cum_max: {cum_max.shape}")
print(f"  breakthrough_diff: {breakthrough_diff.shape}")
print(f"  growth_rate: {growth_rate.shape}")

# Combine all factors
all_temporal_factors = np.hstack([year_factor, year_poly, gen, slide_avg, cum_max, breakthrough_diff, growth_rate])
print(f"All temporal factors combined: {all_temporal_factors.shape}")


Temporal factors created:
  year: (4500, 1)
  year_poly: (4500, 3)
  gen: (4500, 3)
  slide_avg: (4500, 1)
  cum_max: (4500, 1)
  breakthrough_diff: (4500, 2)
  growth_rate: (4500, 1)
All temporal factors combined: (4500, 12)


In [27]:

# Test each temporal factor individually
results_part4 = []

factor_configs = {
    'year_only': year_factor,
    'year_poly': year_poly,
    'tech_gen': gen,
    'slide_avg_pce': slide_avg,
    'cumulative_max': cum_max,
    'breakthrough_diff': breakthrough_diff,
    'growth_rate': growth_rate,
    'all_temporal_12d': all_temporal_factors,
}

# Add the combined: original + all temporal factors
factor_configs['original_all_temporal_118d'] = np.hstack([X_sorted_numeric, all_temporal_factors])

y_train_pt4 = y_sorted_arr[tr_idx]
y_test_pt4 = y_sorted_arr[te_idx]

for factor_name, factors in factor_configs.items():
    # Original features + temporal factor
    if factor_name.startswith('original_'):
        X_comb = factors
    else:
        X_comb = np.hstack([X_sorted_numeric, factors])
    
    X_train, X_test = X_comb[tr_idx], X_comb[te_idx]
    scaler_p4 = StandardScaler()
    X_train_s = scaler_p4.fit_transform(X_train)
    X_test_s = scaler_p4.transform(X_test)
    
    model = LGBMRegressor(n_estimators=200, max_depth=8, learning_rate=0.05, random_state=42, verbosity=-1, n_jobs=4)
    r2, rmse, mae, _ = evaluate_model(model, X_train_s, y_train_pt4, X_test_s, y_test_pt4)
    results_part4.append({
        'factor_name': factor_name, 
        'dim': X_comb.shape[1],
        'r2': r2, 
        'rmse': rmse, 
        'mae': mae
    })
    print(f"{factor_name} (dim={X_comb.shape[1]}): R²={r2:.4f}, RMSE={rmse:.4f}, MAE={mae:.4f}")

# Save Part 4 results
df_part4 = pd.DataFrame(results_part4)
df_part4.to_csv('output/simple_temporal_concat_results.csv', index=False)
print("\n=== Part 4 Complete ===")
print(df_part4.sort_values('r2', ascending=False).to_string())


year_only (dim=107): R²=0.6405, RMSE=3.3742, MAE=2.4608
year_poly (dim=109): R²=0.6405, RMSE=3.3742, MAE=2.4608
tech_gen (dim=109): R²=0.6323, RMSE=3.4124, MAE=2.4771
slide_avg_pce (dim=107): R²=0.6405, RMSE=3.3742, MAE=2.4608
cumulative_max (dim=107): R²=0.6315, RMSE=3.4160, MAE=2.4710
breakthrough_diff (dim=108): R²=0.6405, RMSE=3.3742, MAE=2.4608
growth_rate (dim=107): R²=0.6514, RMSE=3.3227, MAE=2.4135
all_temporal_12d (dim=118): R²=0.6454, RMSE=3.3510, MAE=2.4475
original_all_temporal_118d (dim=118): R²=0.6454, RMSE=3.3510, MAE=2.4475

=== Part 4 Complete ===
                  factor_name  dim        r2      rmse       mae
6                 growth_rate  107  0.651374  3.322709  2.413489
8  original_all_temporal_118d  118  0.645412  3.350998  2.447518
7            all_temporal_12d  118  0.645412  3.350998  2.447518
0                   year_only  107  0.640493  3.374162  2.460784
1                   year_poly  109  0.640493  3.374162  2.460784
3               slide_avg_pce  107  0.6